## Claim Extractor

In [1]:
"""
CASM — Claim Extractor
======================
Stage 1 of the CASM pipeline.

Single responsibility: convert a raw LLM clinical response
into a list of structured, typed, entity-tagged atomic claims.

Does NOT handle patient context or population flags.
Those are handled downstream by the Knowledge Verifier.

Install:
    pip install spacy scispacy
    pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.3/en_core_sci_md-0.5.3.tar.gz
"""

import re
import spacy
from dataclasses import dataclass
from enum import Enum
from typing import Optional


# ─── Enums ────────────────────────────────────────────────────────────────────

class ClaimType(str, Enum):
    DOSAGE_CLAIM      = "DOSAGE_CLAIM"
    DRUG_SAFETY_CLAIM = "DRUG_SAFETY_CLAIM"
    DRUG_INTERACTION  = "DRUG_INTERACTION"
    PROCEDURAL_CLAIM  = "PROCEDURAL_CLAIM"
    DIAGNOSIS_CLAIM   = "DIAGNOSIS_CLAIM"
    POPULATION_CLAIM  = "POPULATION_CLAIM"
    GENERAL_MEDICAL   = "GENERAL_MEDICAL"


# ─── Data Classes ─────────────────────────────────────────────────────────────

@dataclass
class Entity:
    text: str
    label: str    # spaCy entity label e.g. CHEMICAL, DISEASE, DOSAGE
    start: int    # character offset in sentence
    end: int


@dataclass
class Claim:
    claim_id:              str
    claim_text:            str
    claim_type:            ClaimType
    entities:              list[Entity]
    drug_names:            list[str]   # extracted drug/chemical names
    dosages:               list[str]   # extracted dose values e.g. "1000mg"
    conditions:            list[str]   # extracted disease/condition names
    requires_verification: bool        # True for dosage/drug safety/interaction claims
    confidence:            float       # placeholder — filled by Confidence Calibrator later


# ─── Keyword Maps ─────────────────────────────────────────────────────────────

CLAIM_TYPE_KEYWORDS: dict[ClaimType, list[str]] = {
    ClaimType.DOSAGE_CLAIM: [
        "mg", "mcg", "ml", "dose", "dosage", "twice", "once", "three times",
        "daily", "bid", "tid", "qid", "weekly", "monthly", "prescribe",
        "administer", "give", "start", "initiate", "tablet", "capsule",
    ],
    ClaimType.DRUG_SAFETY_CLAIM: [
        "avoid", "contraindicated", "do not use", "caution", "warning",
        "unsafe", "dangerous", "harmful", "risk", "adverse", "side effect",
        "toxicity", "nephrotoxic", "hepatotoxic", "cardiotoxic",
    ],
    ClaimType.DRUG_INTERACTION: [
        "interaction", "interacts", "combined with", "together with",
        "concurrent", "concomitant", "potentiates", "inhibits", "induces",
        "bleeding risk", "increased risk when",
    ],
    ClaimType.PROCEDURAL_CLAIM: [
        "monitor", "check", "test", "measure", "assess", "every", "weeks",
        "months", "follow up", "review", "screen", "scan", "blood test",
        "lab", "hba1c", "creatinine", "egfr", "blood pressure",
    ],
    ClaimType.DIAGNOSIS_CLAIM: [
        "diagnosis", "diagnose", "patient has", "presents with", "suffering from",
        "consistent with", "indicative of", "suggests", "confirms",
    ],
    ClaimType.POPULATION_CLAIM: [
        "elderly", "pediatric", "children", "pregnant", "renal", "kidney",
        "hepatic", "liver", "geriatric", "neonatal", "trimester",
    ],
}

# Conjunctions that can hide multiple claims inside one sentence
SPLIT_CONJUNCTIONS = [
    r"\band also\b",
    r"\badditionally\b",
    r"\bfurthermore\b",
    r"\bin addition\b",
    r"\bmoreover\b",
    r"\bhowever\b",
    r"\balternatively\b",
]


# ─── Extractor ────────────────────────────────────────────────────────────────

class ClaimExtractor:
    """
    Converts a raw LLM clinical response into structured atomic claims.

    Usage:
        extractor = ClaimExtractor()
        claims = extractor.extract("Prescribe Metformin 1000mg twice daily.")
    """

    def __init__(self, model: str = "en_core_sci_md"):
        print(f"[ClaimExtractor] Loading model: {model}")
        self.nlp = spacy.load(model)
        print(f"[ClaimExtractor] Ready.")

    # ── Public ────────────────────────────────────────────────────────────────

    def extract(self, llm_response: str) -> list[Claim]:
        """
        Takes raw LLM response text.
        Returns list of Claim objects.
        No patient context — that is the verifier's job.
        """
        sentences = self._split_into_sentences(llm_response)

        claims = []
        for idx, sentence in enumerate(sentences):
            if not sentence.strip():
                continue
            claim = self._process_sentence(sentence, idx)
            if claim:
                claims.append(claim)

        print(f"[ClaimExtractor] {len(claims)} claims extracted.")
        return claims

    def extract_as_dict(self, llm_response: str) -> list[dict]:
        """Same as extract() but returns JSON-serialisable dicts."""
        return [self._to_dict(c) for c in self.extract(llm_response)]

    # ── Sentence Splitting ────────────────────────────────────────────────────

    def _split_into_sentences(self, text: str) -> list[str]:
        # Primary: spaCy sentence boundary detection
        doc = self.nlp(text)
        sentences = [sent.text.strip() for sent in doc.sents]

        # Secondary: split compound sentences on conjunctions
        final = []
        for sentence in sentences:
            final.extend(self._split_on_conjunctions(sentence))

        # Drop fragments shorter than 4 words
        return [s for s in final if len(s.split()) >= 4]

    def _split_on_conjunctions(self, sentence: str) -> list[str]:
        parts = [sentence]
        for pattern in SPLIT_CONJUNCTIONS:
            new_parts = []
            for part in parts:
                split = re.split(pattern, part, flags=re.IGNORECASE)
                new_parts.extend([s.strip() for s in split if s.strip()])
            parts = new_parts
        return parts

    # ── Claim Processing ──────────────────────────────────────────────────────

    def _process_sentence(self, sentence: str, idx: int) -> Optional[Claim]:
        doc = self.nlp(sentence)

        entities = [
            Entity(text=ent.text, label=ent.label_,
                   start=ent.start_char, end=ent.end_char)
            for ent in doc.ents
        ]

        drug_names = self._extract_drugs(entities, sentence)
        dosages    = self._extract_dosages(entities, sentence)
        conditions = self._extract_conditions(entities, sentence)
        claim_type = self._classify_claim_type(sentence, entities)

        # Drop sentences with no medical content at all
        if claim_type == ClaimType.GENERAL_MEDICAL and not entities:
            return None

        requires_verification = claim_type in [
            ClaimType.DOSAGE_CLAIM,
            ClaimType.DRUG_SAFETY_CLAIM,
            ClaimType.DRUG_INTERACTION,
            ClaimType.POPULATION_CLAIM,
        ]

        return Claim(
            claim_id=f"claim_{idx}",
            claim_text=sentence,
            claim_type=claim_type,
            entities=entities,
            drug_names=drug_names,
            dosages=dosages,
            conditions=conditions,
            requires_verification=requires_verification,
            confidence=0.0,
        )

    # ── Claim Type Classification ─────────────────────────────────────────────

    def _classify_claim_type(self, sentence: str, entities: list[Entity]) -> ClaimType:
        sentence_lower = sentence.lower()
        scores: dict[ClaimType, int] = {ct: 0 for ct in ClaimType}

        for claim_type, keywords in CLAIM_TYPE_KEYWORDS.items():
            for keyword in keywords:
                if keyword in sentence_lower:
                    scores[claim_type] += 1

        best = max(scores, key=lambda ct: scores[ct])
        return best if scores[best] > 0 else ClaimType.GENERAL_MEDICAL

    # ── Entity Extraction ─────────────────────────────────────────────────────

    def _extract_drugs(self, entities: list[Entity], sentence: str) -> list[str]:
        drugs = [e.text for e in entities if e.label in {"CHEMICAL", "DRUG", "MEDICATION"}]
        regex = re.findall(r'\b[A-Z][a-z]+(?:in|ol|ine|ide|ate|mab|nib|pril|sartan|statin)\b', sentence)
        for d in regex:
            if d not in drugs:
                drugs.append(d)
        return list(set(drugs))

    def _extract_dosages(self, entities: list[Entity], sentence: str) -> list[str]:
        dosages = [e.text for e in entities if e.label in {"DOSAGE", "QUANTITY", "CARDINAL"}]
        regex = re.findall(r'\b\d+(?:\.\d+)?\s*(?:mg|mcg|ml|g|units?|IU|mmol)\b', sentence, re.IGNORECASE)
        for d in regex:
            if d.strip() not in dosages:
                dosages.append(d.strip())
        return list(set(dosages))

    def _extract_conditions(self, entities: list[Entity], sentence: str) -> list[str]:
        conditions = [e.text for e in entities if e.label in {"DISEASE", "CONDITION", "DISORDER"}]
        abbrevs = re.findall(r'\b(?:CKD|DM|T2DM|HTN|CAD|CHF|COPD|AF|DVT|PE|UTI)\b', sentence)
        for a in abbrevs:
            if a not in conditions:
                conditions.append(a)
        return list(set(conditions))

    # ── Serialisation ─────────────────────────────────────────────────────────

    def _to_dict(self, claim: Claim) -> dict:
        return {
            "claim_id":              claim.claim_id,
            "claim_text":            claim.claim_text,
            "claim_type":            claim.claim_type.value,
            "entities":              [{"text": e.text, "label": e.label} for e in claim.entities],
            "drug_names":            claim.drug_names,
            "dosages":               claim.dosages,
            "conditions":            claim.conditions,
            "requires_verification": claim.requires_verification,
            "confidence":            claim.confidence,
        }

## Claim Extractor Test

In [2]:
"""
CASM — Claim Extractor Test Suite
==================================
Tests for extractor.py (clean version — no population flags).

Run:
    python test_extractor.py
    pytest test_extractor.py -v
"""

import sys
import json
import inspect

# ─── Setup ────────────────────────────────────────────────────────────────────

def get_extractor():
    try:
        return ClaimExtractor("en_core_sci_md")
    except OSError:
        print("[TEST] Falling back to en_core_web_sm")
        try:
            return ClaimExtractor("en_core_web_sm")
        except OSError:
            print("[TEST] No spaCy model found.")
            print("[TEST] Run: python -m spacy download en_core_web_sm")
            sys.exit(1)


# ─── 1. SENTENCE SPLITTING ────────────────────────────────────────────────────

def test_single_sentence():
    """Single sentence produces one claim."""
    print("\n─── TEST: Single sentence ───")
    e = get_extractor()
    claims = e.extract("Prescribe Metformin 1000mg twice daily.")
    assert len(claims) >= 1, f"Expected 1 claim, got {len(claims)}"
    print(f"  Claims: {len(claims)} ✓ PASS")


def test_multiple_sentences():
    """Three separate sentences produce multiple claims."""
    print("\n─── TEST: Multiple sentences ───")
    e = get_extractor()
    claims = e.extract(
        "Prescribe Metformin 1000mg twice daily. "
        "Avoid Ibuprofen in CKD patients. "
        "Monitor HbA1c every 3 months."
    )
    assert len(claims) >= 2, f"Expected at least 2 claims, got {len(claims)}"
    print(f"  Claims: {len(claims)} ✓ PASS")


def test_conjunction_split_additionally():
    """'Additionally' inside a sentence splits it into two claims."""
    print("\n─── TEST: Conjunction split — additionally ───")
    e = get_extractor()
    claims = e.extract(
        "Prescribe Metformin 1000mg daily. "
        "Additionally monitor kidney function every 3 months."
    )
    assert len(claims) >= 2, f"Expected at least 2 claims, got {len(claims)}"
    print(f"  Claims: {len(claims)} ✓ PASS")


def test_conjunction_split_furthermore():
    """'Furthermore' splits correctly."""
    print("\n─── TEST: Conjunction split — furthermore ───")
    e = get_extractor()
    claims = e.extract(
        "Avoid NSAIDs in renal failure. "
        "Furthermore avoid aminoglycosides in this patient."
    )
    assert len(claims) >= 1
    print(f"  Claims: {len(claims)} ✓ PASS")


def test_conjunction_split_however():
    """'However' splits correctly."""
    print("\n─── TEST: Conjunction split — however ───")
    e = get_extractor()
    claims = e.extract(
        "Metformin is appropriate for T2DM. "
        "However avoid it when eGFR drops below 30."
    )
    assert len(claims) >= 1
    print(f"  Claims: {len(claims)} ✓ PASS")


def test_short_fragment_filtered():
    """Fragments under 4 words are dropped."""
    print("\n─── TEST: Short fragment filtered ───")
    e = get_extractor()
    claims = e.extract("Yes. Prescribe Metformin 1000mg twice daily.")
    texts = [c.claim_text for c in claims]
    assert not any(t.strip() in ["Yes.", "Yes"] for t in texts), \
        "Short fragment should be filtered"
    print(f"  Fragments correctly filtered ✓ PASS")


def test_empty_string_returns_empty_list():
    """Empty input returns empty list without crashing."""
    print("\n─── TEST: Empty input ───")
    e = get_extractor()
    claims = e.extract("")
    assert claims == [], f"Expected [], got {claims}"
    print("  ✓ PASS")


def test_non_medical_text_no_crash():
    """Non-medical text does not crash the extractor."""
    print("\n─── TEST: Non-medical text — no crash ───")
    e = get_extractor()
    claims = e.extract("The weather is nice today. Please come back tomorrow morning.")
    print(f"  Claims from non-medical text: {len(claims)} ✓ PASS (no crash)")


# ─── 2. CLAIM TYPE CLASSIFICATION ────────────────────────────────────────────

def test_dosage_claim_mg_keyword():
    """Sentence with 'mg' classified as DOSAGE_CLAIM."""
    print("\n─── TEST: DOSAGE_CLAIM — mg keyword ───")
    e = get_extractor()
    claims = e.extract("Prescribe Metformin 500mg once daily with meals.")
    assert len(claims) > 0
    assert claims[0].claim_type == ClaimType.DOSAGE_CLAIM, \
        f"Expected DOSAGE_CLAIM, got {claims[0].claim_type}"
    print(f"  Type: {claims[0].claim_type.value} ✓ PASS")


def test_dosage_claim_prescribe_keyword():
    """'Prescribe' keyword triggers DOSAGE_CLAIM."""
    print("\n─── TEST: DOSAGE_CLAIM — prescribe keyword ───")
    e = get_extractor()
    claims = e.extract("Prescribe Lisinopril 10mg to control blood pressure.")
    assert len(claims) > 0
    assert claims[0].claim_type == ClaimType.DOSAGE_CLAIM
    print(f"  Type: {claims[0].claim_type.value} ✓ PASS")


def test_dosage_claim_bid_keyword():
    """'BID' (twice daily abbreviation) triggers DOSAGE_CLAIM."""
    print("\n─── TEST: DOSAGE_CLAIM — BID keyword ───")
    e = get_extractor()
    claims = e.extract("Amoxicillin 500mg BID for 7 days for the infection.")
    assert len(claims) > 0
    print(f"  Type: {claims[0].claim_type.value} ✓ PASS")


def test_drug_safety_claim_avoid():
    """'Avoid' keyword triggers DRUG_SAFETY_CLAIM."""
    print("\n─── TEST: DRUG_SAFETY_CLAIM — avoid ───")
    e = get_extractor()
    claims = e.extract("Avoid NSAIDs in patients with renal impairment.")
    assert len(claims) > 0
    types = [c.claim_type for c in claims]
    assert ClaimType.DRUG_SAFETY_CLAIM in types, \
        f"Expected DRUG_SAFETY_CLAIM in {[t.value for t in types]}"
    print(f"  Types: {[t.value for t in types]} ✓ PASS")


def test_drug_safety_claim_contraindicated():
    """'Contraindicated' triggers DRUG_SAFETY_CLAIM."""
    print("\n─── TEST: DRUG_SAFETY_CLAIM — contraindicated ───")
    e = get_extractor()
    claims = e.extract("Metformin is contraindicated when eGFR falls below 30.")
    assert len(claims) > 0
    types = [c.claim_type for c in claims]
    assert ClaimType.DRUG_SAFETY_CLAIM in types
    print(f"  Types: {[t.value for t in types]} ✓ PASS")


def test_drug_safety_claim_nephrotoxic():
    """'Nephrotoxic' triggers DRUG_SAFETY_CLAIM."""
    print("\n─── TEST: DRUG_SAFETY_CLAIM — nephrotoxic ───")
    e = get_extractor()
    claims = e.extract("Ibuprofen is nephrotoxic and should not be used in CKD.")
    assert len(claims) > 0
    types = [c.claim_type for c in claims]
    assert ClaimType.DRUG_SAFETY_CLAIM in types
    print(f"  Types: {[t.value for t in types]} ✓ PASS")


def test_drug_safety_claim_do_not_use():
    """'Do not use' triggers DRUG_SAFETY_CLAIM."""
    print("\n─── TEST: DRUG_SAFETY_CLAIM — do not use ───")
    e = get_extractor()
    claims = e.extract("Do not use Gentamicin in patients with existing renal disease.")
    assert len(claims) > 0
    types = [c.claim_type for c in claims]
    assert ClaimType.DRUG_SAFETY_CLAIM in types
    print(f"  Types: {[t.value for t in types]} ✓ PASS")


def test_drug_interaction_claim():
    """'Interaction' keyword triggers DRUG_INTERACTION."""
    print("\n─── TEST: DRUG_INTERACTION — interaction keyword ───")
    e = get_extractor()
    claims = e.extract("Ibuprofen interacts with Warfarin and significantly increases bleeding risk.")
    assert len(claims) > 0
    types = [c.claim_type for c in claims]
    assert ClaimType.DRUG_INTERACTION in types, \
        f"Expected DRUG_INTERACTION in {[t.value for t in types]}"
    print(f"  Types: {[t.value for t in types]} ✓ PASS")


def test_drug_interaction_concurrent():
    """'Concurrent' triggers DRUG_INTERACTION."""
    print("\n─── TEST: DRUG_INTERACTION — concurrent ───")
    e = get_extractor()
    claims = e.extract("Concurrent use of MAOIs and SSRIs can cause serotonin syndrome.")
    assert len(claims) > 0
    types = [c.claim_type for c in claims]
    assert ClaimType.DRUG_INTERACTION in types
    print(f"  Types: {[t.value for t in types]} ✓ PASS")


def test_procedural_claim_monitor():
    """'Monitor' triggers PROCEDURAL_CLAIM."""
    print("\n─── TEST: PROCEDURAL_CLAIM — monitor ───")
    e = get_extractor()
    claims = e.extract("Monitor HbA1c and kidney function every 3 months.")
    assert len(claims) > 0
    assert claims[0].claim_type == ClaimType.PROCEDURAL_CLAIM, \
        f"Expected PROCEDURAL_CLAIM, got {claims[0].claim_type}"
    print(f"  Type: {claims[0].claim_type.value} ✓ PASS")


def test_procedural_claim_check():
    """'Check' triggers PROCEDURAL_CLAIM."""
    print("\n─── TEST: PROCEDURAL_CLAIM — check ───")
    e = get_extractor()
    claims = e.extract("Check eGFR levels before initiating Metformin therapy.")
    assert len(claims) > 0
    print(f"  Type: {claims[0].claim_type.value} ✓ PASS")


def test_procedural_claim_follow_up():
    """'Follow up' triggers PROCEDURAL_CLAIM."""
    print("\n─── TEST: PROCEDURAL_CLAIM — follow up ───")
    e = get_extractor()
    claims = e.extract("Follow up with the patient after 4 weeks to review response.")
    assert len(claims) > 0
    print(f"  Type: {claims[0].claim_type.value} ✓ PASS")


def test_diagnosis_claim():
    """'Presents with' triggers DIAGNOSIS_CLAIM."""
    print("\n─── TEST: DIAGNOSIS_CLAIM ───")
    e = get_extractor()
    claims = e.extract("The patient presents with symptoms consistent with Type 2 Diabetes.")
    assert len(claims) > 0
    types = [c.claim_type for c in claims]
    assert ClaimType.DIAGNOSIS_CLAIM in types, \
        f"Expected DIAGNOSIS_CLAIM in {[t.value for t in types]}"
    print(f"  Types: {[t.value for t in types]} ✓ PASS")


def test_population_claim_type():
    """Population keywords in sentence trigger POPULATION_CLAIM type."""
    print("\n─── TEST: POPULATION_CLAIM type ───")
    e = get_extractor()
    claims = e.extract("This dose is appropriate for elderly patients with renal disease.")
    assert len(claims) > 0
    types = [c.claim_type for c in claims]
    has_pop_or_safety = any(t in [
        ClaimType.POPULATION_CLAIM,
        ClaimType.DRUG_SAFETY_CLAIM
    ] for t in types)
    assert has_pop_or_safety
    print(f"  Types: {[t.value for t in types]} ✓ PASS")


# ─── 3. ENTITY EXTRACTION ─────────────────────────────────────────────────────

def test_drug_name_extracted():
    """Drug names extracted from sentence."""
    print("\n─── TEST: Drug name extraction ───")
    e = get_extractor()
    claims = e.extract("Prescribe Metformin 1000mg twice daily.")
    assert len(claims) > 0
    has_content = len(claims[0].drug_names) > 0 or len(claims[0].dosages) > 0
    assert has_content, "Expected drug names or dosages extracted"
    print(f"  Drugs: {claims[0].drug_names}, Dosages: {claims[0].dosages} ✓ PASS")


def test_dosage_mg_extracted():
    """Dosage in mg extracted by regex."""
    print("\n─── TEST: Dosage extraction — Xmg ───")
    e = get_extractor()
    claims = e.extract("Prescribe Ibuprofen 400mg three times daily.")
    assert len(claims) > 0
    print(f"  Dosages: {claims[0].dosages} ✓ PASS")


def test_dosage_decimal_extracted():
    """Decimal dosages like 2.5mg extracted."""
    print("\n─── TEST: Dosage extraction — decimal ───")
    e = get_extractor()
    claims = e.extract("Give Warfarin 2.5mg once daily at the same time each evening.")
    assert len(claims) > 0
    print(f"  Dosages: {claims[0].dosages} ✓ PASS")


def test_dosage_mcg_extracted():
    """mcg unit extracted correctly."""
    print("\n─── TEST: Dosage extraction — mcg ───")
    e = get_extractor()
    claims = e.extract("Prescribe Levothyroxine 50mcg once daily in the morning.")
    assert len(claims) > 0
    print(f"  Dosages: {claims[0].dosages} ✓ PASS")


def test_condition_ckd_abbreviation():
    """CKD abbreviation extracted as condition."""
    print("\n─── TEST: Condition — CKD abbreviation ───")
    e = get_extractor()
    claims = e.extract("This drug is unsafe in patients with CKD.")
    assert len(claims) > 0
    all_conditions = [c for claim in claims for c in claim.conditions]
    print(f"  Conditions: {all_conditions} ✓ PASS")


def test_condition_t2dm_abbreviation():
    """T2DM abbreviation extracted."""
    print("\n─── TEST: Condition — T2DM abbreviation ───")
    e = get_extractor()
    claims = e.extract("First line therapy for T2DM is Metformin 500mg twice daily.")
    assert len(claims) > 0
    all_conditions = [c for claim in claims for c in claim.conditions]
    print(f"  Conditions: {all_conditions} ✓ PASS")


def test_condition_af_abbreviation():
    """AF abbreviation extracted."""
    print("\n─── TEST: Condition — AF abbreviation ───")
    e = get_extractor()
    claims = e.extract("Warfarin is indicated for AF to reduce stroke risk.")
    assert len(claims) > 0
    all_conditions = [c for claim in claims for c in claim.conditions]
    print(f"  Conditions: {all_conditions} ✓ PASS")


def test_multiple_drugs_in_sentence():
    """Both drug names extracted from interaction sentence."""
    print("\n─── TEST: Multiple drugs in one sentence ───")
    e = get_extractor()
    claims = e.extract("Ibuprofen interacts with Warfarin causing elevated bleeding risk.")
    assert len(claims) > 0
    print(f"  Drugs: {claims[0].drug_names} ✓ PASS")


# ─── 4. REQUIRES_VERIFICATION ────────────────────────────────────────────────

def test_dosage_requires_verification_true():
    """DOSAGE_CLAIM: requires_verification = True."""
    print("\n─── TEST: requires_verification — DOSAGE_CLAIM ───")
    e = get_extractor()
    claims = e.extract("Prescribe Glipizide 5mg daily for blood sugar control.")
    dosage = [c for c in claims if c.claim_type == ClaimType.DOSAGE_CLAIM]
    if dosage:
        assert dosage[0].requires_verification is True
        print(f"  requires_verification: True ✓ PASS")
    else:
        print("  No DOSAGE_CLAIM extracted (model-dependent) ✓ PASS")


def test_drug_safety_requires_verification_true():
    """DRUG_SAFETY_CLAIM: requires_verification = True."""
    print("\n─── TEST: requires_verification — DRUG_SAFETY_CLAIM ───")
    e = get_extractor()
    claims = e.extract("Avoid Ibuprofen in patients with renal failure.")
    safety = [c for c in claims if c.claim_type == ClaimType.DRUG_SAFETY_CLAIM]
    if safety:
        assert safety[0].requires_verification is True
        print(f"  requires_verification: True ✓ PASS")
    else:
        print("  No DRUG_SAFETY_CLAIM extracted ✓ PASS")


def test_interaction_requires_verification_true():
    """DRUG_INTERACTION: requires_verification = True."""
    print("\n─── TEST: requires_verification — DRUG_INTERACTION ───")
    e = get_extractor()
    claims = e.extract("Warfarin and Aspirin concurrent use increases bleeding risk.")
    interaction = [c for c in claims if c.claim_type == ClaimType.DRUG_INTERACTION]
    if interaction:
        assert interaction[0].requires_verification is True
        print(f"  requires_verification: True ✓ PASS")
    else:
        print("  No DRUG_INTERACTION extracted ✓ PASS")


def test_procedural_requires_verification_false():
    """PROCEDURAL_CLAIM: requires_verification = False."""
    print("\n─── TEST: requires_verification — PROCEDURAL_CLAIM = False ───")
    e = get_extractor()
    claims = e.extract("Monitor blood pressure every week during dose titration.")
    procedural = [c for c in claims if c.claim_type == ClaimType.PROCEDURAL_CLAIM]
    if procedural:
        assert procedural[0].requires_verification is False
        print(f"  requires_verification: False ✓ PASS")
    else:
        print("  No PROCEDURAL_CLAIM extracted ✓ PASS")


# ─── 5. ARCHITECTURE CONTRACTS ───────────────────────────────────────────────

def test_extract_signature_no_patient_context():
    """extract() must NOT accept patient_context — belongs to verifier."""
    print("\n─── TEST: extract() signature — no patient_context ───")
    e = get_extractor()
    sig = inspect.signature(e.extract)
    params = list(sig.parameters.keys())
    assert "patient_context" not in params, \
        f"patient_context should not be in extract() params: {params}"
    assert "llm_response" in params
    print(f"  Parameters: {params} ✓ PASS")


def test_claim_has_no_population_flags_field():
    """Claim dataclass must not have population_flags field."""
    print("\n─── TEST: Claim has no population_flags field ───")
    e = get_extractor()
    claims = e.extract("Prescribe Metformin 500mg daily.")
    if claims:
        assert not hasattr(claims[0], "population_flags"), \
            "population_flags must not exist on Claim — belongs to verifier"
    print("  No population_flags field on Claim ✓ PASS")


def test_confidence_defaults_to_zero():
    """Confidence must be 0.0 — filled by Confidence Calibrator later."""
    print("\n─── TEST: confidence defaults to 0.0 ───")
    e = get_extractor()
    claims = e.extract("Prescribe Metformin 1000mg twice daily.")
    for c in claims:
        assert c.confidence == 0.0, f"Expected 0.0, got {c.confidence}"
    print("  confidence = 0.0 ✓ PASS")


def test_claim_ids_sequential():
    """Claim IDs should follow claim_0, claim_1, claim_2 format."""
    print("\n─── TEST: Claim IDs are sequential ───")
    e = get_extractor()
    claims = e.extract(
        "Prescribe Metformin 500mg daily. "
        "Avoid NSAIDs in kidney disease. "
        "Monitor creatinine every 3 months."
    )
    for claim in claims:
        assert claim.claim_id.startswith("claim_"), f"Bad ID format: {claim.claim_id}"
    print(f"  IDs: {[c.claim_id for c in claims]} ✓ PASS")


# ─── 6. SERIALISATION ────────────────────────────────────────────────────────

def test_extract_as_dict_json_serialisable():
    """extract_as_dict() output must be fully JSON serialisable."""
    print("\n─── TEST: extract_as_dict() is JSON serialisable ───")
    e = get_extractor()
    result = e.extract_as_dict("Prescribe Metformin 1000mg daily. Avoid NSAIDs in CKD.")
    json_str = json.dumps(result)
    parsed = json.loads(json_str)
    assert isinstance(parsed, list) and len(parsed) > 0
    print(f"  Serialised {len(parsed)} claims ✓ PASS")


def test_dict_has_all_required_keys():
    """Claim dict must have all 9 required keys."""
    print("\n─── TEST: Dict has all required keys ───")
    e = get_extractor()
    result = e.extract_as_dict("Prescribe Metformin 500mg twice daily.")
    required = {
        "claim_id", "claim_text", "claim_type", "entities",
        "drug_names", "dosages", "conditions",
        "requires_verification", "confidence"
    }
    for claim in result:
        missing = required - set(claim.keys())
        assert not missing, f"Missing keys: {missing}"
    print(f"  All 9 keys present ✓ PASS")


def test_dict_has_no_population_flags_key():
    """Claim dict must NOT have population_flags key."""
    print("\n─── TEST: Dict has no population_flags key ───")
    e = get_extractor()
    result = e.extract_as_dict("Prescribe Metformin 1000mg daily.")
    for claim in result:
        assert "population_flags" not in claim, \
            "population_flags must not be in claim dict — belongs to verifier"
    print("  No population_flags in dict ✓ PASS")


def test_claim_type_is_string_not_enum():
    """claim_type in dict must be a plain string, not an Enum object."""
    print("\n─── TEST: claim_type is string in dict ───")
    e = get_extractor()
    result = e.extract_as_dict("Prescribe Metformin 500mg twice daily.")
    if result:
        assert isinstance(result[0]["claim_type"], str), \
            f"Expected str, got {type(result[0]['claim_type'])}"
    print("  claim_type is str ✓ PASS")


def test_entities_are_list_of_dicts():
    """entities in dict must be a list of dicts with text and label."""
    print("\n─── TEST: entities format in dict ───")
    e = get_extractor()
    result = e.extract_as_dict("Prescribe Metformin 500mg for Type 2 Diabetes.")
    if result:
        entities = result[0]["entities"]
        assert isinstance(entities, list)
        for ent in entities:
            assert "text" in ent and "label" in ent
    print("  entities format correct ✓ PASS")


# ─── 7. CORE CASM SCENARIO ───────────────────────────────────────────────────

def test_core_casm_scenario():
    """
    THE MAIN TEST.
    72yo diabetic CKD patient on Warfarin.
    Validates all 4 dangerous claims are extracted correctly.
    Population flags NOT tested here — that is the verifier's job.
    """
    print("\n" + "═" * 58)
    print("  CORE TEST: 72yo CKD Diabetic on Warfarin")
    print("═" * 58)

    e = get_extractor()

    llm_response = (
        "For blood sugar control, prescribe Metformin 1000mg twice daily. "
        "For pain management, Ibuprofen 400mg three times daily is appropriate. "
        "Add Glipizide 5mg daily to quickly bring down HbA1c. "
        "Monitor kidney function every 6 months."
    )

    claims = e.extract(llm_response)

    print(f"\n  Input: {llm_response}\n")
    print(f"  Extracted {len(claims)} claims:\n")
    for c in claims:
        marker = "⚠️  VERIFY" if c.requires_verification else "✓  ok"
        print(f"  [{c.claim_id}] {c.claim_type.value}")
        print(f"         Text:    {c.claim_text}")
        print(f"         Drugs:   {c.drug_names}")
        print(f"         Dosages: {c.dosages}")
        print(f"         Status:  {marker}")
        print()

    # Core assertions
    assert len(claims) >= 3, f"Expected ≥3 claims, got {len(claims)}"

    needs_verify = [c for c in claims if c.requires_verification]
    assert len(needs_verify) >= 2, \
        f"Expected ≥2 claims requiring verification, got {len(needs_verify)}"

    assert all(c.confidence == 0.0 for c in claims), \
        "All confidence scores should be 0.0 at extraction stage"

    assert not any(hasattr(c, "population_flags") for c in claims), \
        "population_flags must not exist on Claim at extraction stage"

    print(f"  ✓ {len(claims)} claims extracted")
    print(f"  ✓ {len(needs_verify)} require verification")
    print(f"  ✓ confidence = 0.0 on all")
    print(f"  ✓ no population_flags on claims")
    print("  ✓ CORE TEST PASSED")


# ─── Runner ───────────────────────────────────────────────────────────────────

TESTS = [
    test_single_sentence,
    test_multiple_sentences,
    test_conjunction_split_additionally,
    test_conjunction_split_furthermore,
    test_conjunction_split_however,
    test_short_fragment_filtered,
    test_empty_string_returns_empty_list,
    test_non_medical_text_no_crash,
    test_dosage_claim_mg_keyword,
    test_dosage_claim_prescribe_keyword,
    test_dosage_claim_bid_keyword,
    test_drug_safety_claim_avoid,
    test_drug_safety_claim_contraindicated,
    test_drug_safety_claim_nephrotoxic,
    test_drug_safety_claim_do_not_use,
    test_drug_interaction_claim,
    test_drug_interaction_concurrent,
    test_procedural_claim_monitor,
    test_procedural_claim_check,
    test_procedural_claim_follow_up,
    test_diagnosis_claim,
    test_population_claim_type,
    test_drug_name_extracted,
    test_dosage_mg_extracted,
    test_dosage_decimal_extracted,
    test_dosage_mcg_extracted,
    test_condition_ckd_abbreviation,
    test_condition_t2dm_abbreviation,
    test_condition_af_abbreviation,
    test_multiple_drugs_in_sentence,
    test_dosage_requires_verification_true,
    test_drug_safety_requires_verification_true,
    test_interaction_requires_verification_true,
    test_procedural_requires_verification_false,
    test_extract_signature_no_patient_context,
    test_claim_has_no_population_flags_field,
    test_confidence_defaults_to_zero,
    test_claim_ids_sequential,
    test_extract_as_dict_json_serialisable,
    test_dict_has_all_required_keys,
    test_dict_has_no_population_flags_key,
    test_claim_type_is_string_not_enum,
    test_entities_are_list_of_dicts,
    test_core_casm_scenario,
]


if __name__ == "__main__":
    passed = failed = 0

    for test in TESTS:
        try:
            test()
            passed += 1
        except AssertionError as err:
            print(f"  ✗ FAIL: {err}")
            failed += 1
        except Exception as err:
            print(f"  ✗ ERROR in {test.__name__}: {type(err).__name__}: {err}")
            failed += 1

    print(f"\n{'═' * 58}")
    print(f"  {passed} passed  /  {failed} failed  /  {len(TESTS)} total")
    print(f"{'═' * 58}")
    sys.exit(0 if failed == 0 else 1)


─── TEST: Single sentence ───
[ClaimExtractor] Loading model: en_core_sci_md


C:\Users\basim\PycharmProjects\PythonProject\.venv\Lib\site-packages\spacy\language.py:2195: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]


[ClaimExtractor] Ready.
[ClaimExtractor] 1 claims extracted.
  Claims: 1 ✓ PASS

─── TEST: Multiple sentences ───
[ClaimExtractor] Loading model: en_core_sci_md
[ClaimExtractor] Ready.
[ClaimExtractor] 3 claims extracted.
  Claims: 3 ✓ PASS

─── TEST: Conjunction split — additionally ───
[ClaimExtractor] Loading model: en_core_sci_md
[ClaimExtractor] Ready.
[ClaimExtractor] 2 claims extracted.
  Claims: 2 ✓ PASS

─── TEST: Conjunction split — furthermore ───
[ClaimExtractor] Loading model: en_core_sci_md
[ClaimExtractor] Ready.
[ClaimExtractor] 2 claims extracted.
  Claims: 2 ✓ PASS

─── TEST: Conjunction split — however ───
[ClaimExtractor] Loading model: en_core_sci_md
[ClaimExtractor] Ready.
[ClaimExtractor] 2 claims extracted.
  Claims: 2 ✓ PASS

─── TEST: Short fragment filtered ───
[ClaimExtractor] Loading model: en_core_sci_md
[ClaimExtractor] Ready.
[ClaimExtractor] 1 claims extracted.
  Fragments correctly filtered ✓ PASS

─── TEST: Empty input ───
[ClaimExtractor] Loading mod

SystemExit: 0

C:\Users\basim\PycharmProjects\PythonProject\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3709: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
